In [2]:
# ==============================================================================
# NOTEBOOK 01: FEATURE EXTRACTION
# ==============================================================================
# OBIETTIVO:
# 1. Pre-processare ogni file audio del dataset UrbanSound8K per renderlo
#    standard e robusto.
# 2. Estrarre un set completo di feature audio (2D e 1D).
# 3. Salvare le feature in file .npz per un caricamento rapido nei notebook
#    di addestramento, mantenendo la struttura originale dei 10 fold.
#
# Questo notebook va eseguito una sola volta all'inizio del progetto.
# ==============================================================================

import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

# --- 1. CONFIGURAZIONE DEI PERCORSI E PARAMETRI ---

# Percorsi principali
DATA_ROOT = "../data"
RAW_DATA_PATH = os.path.join(DATA_ROOT, "raw")
PROCESSED_DATA_PATH = os.path.join(DATA_ROOT, "processed", "features")
META_FILE_PATH = os.path.join(RAW_DATA_PATH, "UrbanSound8K.csv")

# Parametri audio
TARGET_SR = 22050  # Frequenza di campionamento standard per molti modelli audio
TARGET_LENGTH_S = 4  # Durata fissa di ogni clip in secondi

print("Configurazione completata:")
print(f"  - Percorso dati grezzi: {RAW_DATA_PATH}")
print(f"  - Percorso dati processati: {PROCESSED_DATA_PATH}")
print(f"  - Frequenza di campionamento target: {TARGET_SR} Hz")
print(f"  - Durata target: {TARGET_LENGTH_S} secondi")

# --- 2. FUNZIONI DI PRE-PROCESSING AUDIO ---

def load_and_standardize_audio(audio_path, target_sr):
    """
    Carica un file audio e applica le standardizzazioni di base.
    - Carica il file audio.
    - Converte in mono (fondamentale per avere un solo canale).
    - Normalizza l'ampiezza di picco a 1.0 (comportamento di default di librosa).
    - Esegue il resampling alla frequenza di campionamento target.
    """
    try:
        y, sr = librosa.load(audio_path, sr=None, mono=True)
        
        if sr != target_sr:
            y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
            
        return y, target_sr
    except Exception as e:
        print(f"Errore durante il caricamento e la standardizzazione di {os.path.basename(audio_path)}: {e}")
        return None, None

def enhance_and_frame_audio(y, target_sr, target_duration_s, trim_db=25):
    """
    Applica tecniche di miglioramento e uniforma la lunghezza della clip.
    - Rimozione dei silenzi: centra l'analisi sulla parte più "attiva" della clip.
    - Framing: assicura che ogni clip abbia la stessa lunghezza, prendendo la
      porzione centrale se più lunga, o aggiungendo padding simmetrico se più corta.
    """
    target_length_samples = int(target_sr * target_duration_s)

    # Rimozione dei silenzi all'inizio e alla fine della clip
    y, _ = librosa.effects.trim(y, top_db=trim_db)
    
    # Framing con centratura
    if len(y) > target_length_samples:
        # La clip è più lunga del target: prendiamo la porzione centrale
        start_offset = (len(y) - target_length_samples) // 2
        y = y[start_offset : start_offset + target_length_samples]
    else:
        # La clip è più corta del target: applichiamo padding simmetrico
        pad_width = target_length_samples - len(y)
        pad_left = pad_width // 2
        pad_right = pad_width - pad_left
        y = np.pad(y, (pad_left, pad_right), mode='constant')
        
    return y

# --- 3. FUNZIONE DI ESTRAZIONE DELLE FEATURE ---

def extract_all_features(y, sr):
    """
    Estrae un set completo di feature dall'array audio pre-processato.
    Include feature 2D (simili a immagini) e 1D (serie temporali).
    """
    features = {}
    
    # Feature 2D (Tempo-Frequenza)
    features['log_mel_spec'] = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128))
    features['mfcc'] = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    features['chroma'] = librosa.feature.chroma_stft(y=y, sr=sr)
    features['contrast'] = librosa.feature.spectral_contrast(y=y, sr=sr)
    
    # Feature 1D (Vettori temporali) - appiattite per coerenza
    features['centroid'] = librosa.feature.spectral_centroid(y=y, sr=sr).flatten()
    features['rolloff'] = librosa.feature.spectral_rolloff(y=y, sr=sr).flatten()
    features['zcr'] = librosa.feature.zero_crossing_rate(y).flatten()
    
    return features

# --- 4. ESECUZIONE DEL WORKFLOW COMPLETO ---

print("\nAvvio del processo di estrazione delle feature per l'intero dataset...")

df_meta = pd.read_csv(META_FILE_PATH)
print(f"Trovati {len(df_meta)} metadati di file audio da processare.")

processed_count = 0
error_count = 0

for _, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
    fold_num = row['fold']
    filename = row['slice_file_name']
    
    # Definizione percorsi di input e output
    src_path = os.path.join(RAW_DATA_PATH, f"fold{fold_num}", filename)
    dst_dir = os.path.join(PROCESSED_DATA_PATH, f"fold{fold_num}")
    os.makedirs(dst_dir, exist_ok=True)
    out_name = os.path.splitext(filename)[0]
    dst_path = os.path.join(dst_dir, f"{out_name}.npz")
    
    # Salta i file già processati per permettere di riprendere il processo
    if os.path.exists(dst_path):
        processed_count += 1
        continue
        
    try:
        # Step 1: Caricamento e standardizzazione
        y_raw, sr = load_and_standardize_audio(src_path, TARGET_SR)
        if y_raw is None:
            error_count += 1
            continue
            
        # Step 2: Miglioramento e framing
        y_processed = enhance_and_frame_audio(y_raw, sr, TARGET_LENGTH_S)
        
        # Step 3: Estrazione delle feature
        features = extract_all_features(y_processed, sr)
        
        # Step 4: Salvataggio in formato .npz compresso
        np.savez_compressed(
            dst_path,
            log_mel_spec=features['log_mel_spec'],
            mfcc=features['mfcc'],
            chroma=features['chroma'],
            contrast=features['contrast'],
            centroid=features['centroid'],
            rolloff=features['rolloff'],
            zcr=features['zcr'],
            class_id=row['classID']
        )
        processed_count += 1
        
    except Exception as e:
        print(f"Errore irreversibile processando {filename}: {e}")
        error_count += 1

print("\n--- Processo di Estrazione Completato ---")
print(f"File processati e salvati con successo: {processed_count}")
print(f"Errori riscontrati: {error_count}")
print(f"I dati sono ora pronti in: {PROCESSED_DATA_PATH}")

Configurazione completata:
  - Percorso dati grezzi: ../data\raw
  - Percorso dati processati: ../data\processed\features
  - Frequenza di campionamento target: 22050 Hz
  - Durata target: 4 secondi

Avvio del processo di estrazione delle feature per l'intero dataset...
Trovati 8732 metadati di file audio da processare.


  0%|          | 0/8732 [00:00<?, ?it/s]


--- Processo di Estrazione Completato ---
File processati e salvati con successo: 8732
Errori riscontrati: 0
I dati sono ora pronti in: ../data\processed\features
